# Streaming Multivariate Drift Tutorial

This notebook demonstrates `StreamMonitor` with adaptive thresholds and alert sinks in a streaming-style loop.

In [ ]:
import asyncio
import numpy as np
import pandas as pd
from drift_control.alert_sinks import LogAlertSink
from drift_control.stream_monitor import StreamMonitor
from drift_control.psi_drift_detector import PSIDriftDetector

In [ ]:
rng = np.random.default_rng(7)
baseline_df = pd.DataFrame({
    'amount': rng.normal(100, 20, size=500),
    'latency_ms': rng.normal(250, 35, size=500),
})
baseline_df.head()

In [ ]:
detector = PSIDriftDetector(threshold=0.2)
monitor = StreamMonitor(
    detector=detector,
    baseline_strategy='sliding',
    sliding_window_batches=3,
    adaptive_threshold=True,
    threshold_quantile=0.95,
    threshold_history=50,
    min_threshold_samples=10,
    alert_sinks=[LogAlertSink()],
)
monitor.set_baseline(baseline_df)

In [ ]:
batches = [
    pd.DataFrame({'amount': rng.normal(102, 20, size=120), 'latency_ms': rng.normal(248, 35, size=120)}),
    pd.DataFrame({'amount': rng.normal(104, 20, size=120), 'latency_ms': rng.normal(247, 35, size=120)}),
    pd.DataFrame({'amount': rng.normal(125, 25, size=120), 'latency_ms': rng.normal(290, 45, size=120)}),
]

async def stream():
    for b in batches:
        yield b

results = []
async def run():
    async for res in monitor.monitor(stream()):
        results.append(res)

asyncio.run(run())
results

## What to operationalize next

- Replace `LogAlertSink` with `SlackWebhookAlertSink` or `PagerDutyAlertSink`.
- Persist stream outputs and alert events for incident analysis.
- Use `ColumnFilterAlertSink` to escalate only high-risk features.